In [33]:
#this prints all the data to see what we are dealing with
from astropy.io import fits
import os
import re
from datetime import datetime
path = '/Users/jesuslongares/Desktop/GORT_Archive_DB/GORT_FITS'
# open through the folder and find the folder that holds the fits files
for file in sorted(os.listdir(path)):
    fpath = os.path.join(path, file)
    # skips any of the file that is not fits
    if (not os.path.isfile(fpath)) or (not file.lower().endswith(('.fits', '.fit', '.fts'))):
        continue
    with fits.open(fpath) as hdul:
        print(f'================={file}=================')
       #let's work on creating the spreadsheet that holds all the header files and any futher information
        header = hdul[0].header
        for key, value in header.items():
            print(f'{key}: {value}')

        #parse through the file name and break into parts
        name = file.replace('.fts', '').replace('.fits', '')
        parts = name.split('-')
        if len(parts) > 1 and '@' in parts[1]:
            date_time = parts[1].split('@')
            date = date_time[0]
            time = date_time[1]
            dt = datetime.strptime(date + time, "%Y%m%d%H%M%S")

        for i, part in enumerate(parts):
            print(f'{i}: {part}')
        
    

=================DRACO DWARF-20230722@051431-300-001-E-Clear.fts=================
SIMPLE: True
BITPIX: -32
NAXIS: 2
NAXIS1: 341
NAXIS2: 341
BSCALE: 1.0
BZERO: 0.0
DATE-OBS: 2023-07-22T05:14:48.780
EXPTIME: 300.0
EXPOSURE: 300.0
SET-TEMP: -25.0
CCD-TEMP: -16.15944825
XPIXSZ: 39.0
YPIXSZ: 39.0
XBINNING: 3
YBINNING: 3
XORGSUBF: 0
YORGSUBF: 0
READOUTM: Monochrome
FILTER: Clear
IMAGETYP: Light Frame
FOCALLEN: 3984.8
APTDIA: 356.0
APTAREA: 89345.50974291521
SBSTDVER: SBFITSEXT Version 1.0
SWCREATE: MaxIm DL Version 6.29 220209 053S4
SWSERIAL: 053S4-7FE7Y-W7ATY-313X2-A6RJY-1V
FOCUSSSZ: 10.0
SITELAT: 38 33 57
SITELONG: -122 41 14
JD: 2460147.7186201387
JD-HELIO: 2460147.7231887705
OBJECT: DRACO DWARF
TELESCOP: Celestron 14
INSTRUME: Apogee USB/Net
OBSERVER: GORT Staff
NOTES: 
ROWORDER: TOP-DOWN
FLIPSTAT: 
CSTRETCH: Medium
CBLACK: 11433
CWHITE: 20853
PEDESTAL: 0
SWOWNER: David Kensiski
JD-OBS: 2460147.7186201
HJD-OBS: 2460147.7190952
BJD-OBS: 2460147.7199472
OBJCTAZ: 6.8758
AZIMUTH: 6.8758
OBJC

In [37]:
from astropy.io import fits
import os

path = '/Users/jesuslongares/Desktop/GORT_Archive_DB/GORT_FITS'

for file in sorted(os.listdir(path)):
    fpath = os.path.join(path, file)

    if (not os.path.isfile(fpath)) or (not file.lower().endswith(('.fits', '.fit', '.fts'))):
        continue

    try:
        with fits.open(fpath) as hdul:
            
            print(f"\n===== {file} =====")
            print(hdul[0].header.tostring(sep="\n"))

    except OSError as e:
        print(f"\nSKIPPED (not valid FITS): {file}")
        print("Reason:", e)


===== DRACO DWARF-20230722@051431-300-001-E-Clear.fts =====
SIMPLE  =                    T                                                  
BITPIX  =                  -32 /8 unsigned int, 16 & 32 int, -32 & -64 real     
NAXIS   =                    2 /number of axes                                  
NAXIS1  =                  341 /fastest changing axis                           
NAXIS2  =                  341 /next to fastest changing axis                   
BSCALE  =   1.0000000000000000 /physical = BZERO + BSCALE*array_value           
BZERO   =   0.0000000000000000 /physical = BZERO + BSCALE*array_value           
DATE-OBS= '2023-07-22T05:14:48.780' / [ISO 8601] UTC date/time of exposure start
EXPTIME =   3.00000000000E+002 / [sec] Duration of exposure                     
EXPOSURE=   3.00000000000E+002 / [sec] Duration of exposure                     
SET-TEMP=  -25.000000000000000 /CCD temperature setpoint in C                   
CCD-TEMP=  -16.159448250000001 /CCD temperature 

In [17]:
with fits.open('/Users/jesuslongares/Downloads/M 101-20230813@061826-090-001-W-R.fts') as hdul:
    hdul.info()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/jesuslongares/Downloads/M 101-20230813@061826-090-001-W-R.fts'

In [39]:
#goes through the fits files and generates the csv files with all the data given within the fits
from astropy.io import fits
import os
import pandas as pd

path = '/Users/jesuslongares/Desktop/GORT_Archive_DB/GORT_FITS'

rows = []
all_keys = set()

for file in sorted(os.listdir(path)):
    fpath = os.path.join(path, file)

    if os.path.isfile(fpath) and file.lower().endswith(('.fits', '.fit', '.fts')):
        with fits.open(fpath) as hdul:
            hdr = hdul[0].header

            d = {"FILENAME": file}
            for k, v in hdr.items():
                if isinstance(v, (str, int, float, bool)) or v is None:
                    d[k] = v
                else:
                    d[k] = str(v)
            rows.append(d)
            all_keys.update(d.keys())

columns = ["FILENAME"] + sorted(k for k in all_keys if k != "FILENAME")
df = pd.DataFrame(rows, columns=columns)
out_path = '/Users/jesuslongares/Desktop/GORT_Archive_DB/'
out_csv = os.path.join(path, "gort_all_headers.csv")
df.to_csv(out_csv, index=False)

print("Saved:", out_csv)

Saved: /Users/jesuslongares/Desktop/GORT_Archive_DB/GORT_FITS/gort_all_headers.csv
